# Synthetic Radio Host Generator

In [ ]:
import os
import sys
import subprocess
import logging

# ------------------ CONFIG ------------------
REPO_URL = "https://github.com/Procrastinator02/synthetic_radio_host.git"
REPO_DIR = "/content/synthetic_radio_host"
BRANCH_NAME = "master"   # 🔁 change this to your branch
# --------------------------------------------

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)
logger = logging.getLogger(__name__)

def run_cmd(cmd, cwd=None):
    """Run shell command safely and fail fast"""
    logger.info(f"Running: {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    if result.returncode != 0:
        logger.error(result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result.stdout.strip()

# Clone or update repository
if not os.path.exists(REPO_DIR):
    logger.info(f"Cloning branch '{BRANCH_NAME}' from {REPO_URL}")
    run_cmd([
        "git", "clone",
        "--branch", BRANCH_NAME,
        "--single-branch",
        REPO_URL,
        REPO_DIR
    ])
    logger.info("Repository cloned successfully")
else:
    logger.info("Repository exists, updating branch")
    run_cmd(["git", "fetch", "origin"], cwd=REPO_DIR)
    run_cmd(["git", "checkout", BRANCH_NAME], cwd=REPO_DIR)
    run_cmd(["git", "pull", "origin", BRANCH_NAME], cwd=REPO_DIR)
    logger.info("Repository updated successfully")

# Add repo to PYTHONPATH
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    logger.info(f"Added {REPO_DIR} to Python path")


2026-01-02 17:13:37 - INFO - Repository exists, updating branch
2026-01-02 17:13:37 - INFO - Running: git fetch origin
2026-01-02 17:13:38 - INFO - Running: git checkout elevenlabs
2026-01-02 17:13:38 - INFO - Running: git pull origin elevenlabs
2026-01-02 17:13:39 - INFO - Repository updated successfully
2026-01-02 17:13:39 - INFO - Added /content/synthetic_radio_host to Python path


In [2]:
# ------------------ CONFIG ------------------
REPO_SRC_DIR = "/content/synthetic_radio_host/src"
REQUIREMENTS_FILE = "/content/synthetic_radio_host/requirements.txt"
DEPS_MARKER = "/content/.deps_installed"
# --------------------------------------------

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)
logger = logging.getLogger(__name__)

def run_cmd(cmd):
    """Run shell command safely and fail fast"""
    logger.info(f"Running: {' '.join(cmd)}")
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    if result.returncode != 0:
        logger.error(result.stderr)
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result.stdout.strip()

# ---------------- Dependency Installation ----------------
if not os.path.exists(DEPS_MARKER):
    logger.info("Checking requirements.txt")

    if not os.path.exists(REQUIREMENTS_FILE):
        raise FileNotFoundError(f"Missing {REQUIREMENTS_FILE}")

    logger.info("Installing dependencies from requirements.txt")
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "--upgrade", "pip"
    ])
    run_cmd([
        sys.executable, "-m", "pip", "install",
        "-r", REQUIREMENTS_FILE
    ])

    open(DEPS_MARKER, "w").close()
    logger.info("Dependencies installed successfully")
else:
    logger.info("Dependencies already installed — skipping")

# ---------------- PYTHONPATH Setup ----------------
if REPO_SRC_DIR not in sys.path:
    sys.path.insert(0, REPO_SRC_DIR)
    logger.info(f"Added {REPO_SRC_DIR} to Python path")
else:
    logger.info("src/ already present in Python path")

2026-01-02 17:13:39 - INFO - Dependencies already installed — skipping
2026-01-02 17:13:39 - INFO - Added /content/synthetic_radio_host/src to Python path


# Setup and Configuration

In [ ]:
from radio_host_functions import (
    CONFIG,
    __version__,
    fetch_wikipedia_article,
    generate_script_prompt,
    generate_script,
    generate_audio_segments,
    combine_and_export_audio
)

import google.generativeai as genai
from google.colab import userdata, files
import os

# ---------------- Logging ----------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True
)
logger = logging.getLogger(__name__)

# ---------------- API Keys ----------------
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY not found in Colab secrets")
    
    # Set Gemini API key (used for both script generation and TTS)
    genai.configure(api_key=GEMINI_API_KEY)
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    logger.info("Gemini API key configured successfully")

except Exception as e:
    logger.error(f"Failed to load API keys: {e}")
    raise

# ---------------- Gemini Model ----------------
gemini_model = genai.GenerativeModel(CONFIG["GEMINI_MODEL"])
logger.info(f"Gemini model '{CONFIG['GEMINI_MODEL']}' initialized")

logger.info(f"Synthetic Radio Host version: {__version__}")


ModuleNotFoundError: No module named 'elevenlabs.client'

In [ ]:
# This cell is no longer needed - initialization is done in Cell 5
# Keeping it for reference but it's redundant now
logger.info("All API clients already initialized in previous cell")


In [ ]:
# ---------------- Config ----------------
WIKIPEDIA_TOPIC = "MS Dhoni"
MAX_LOG_CHARS = 300  # prevent noisy logs
# ---------------------------------------

logger.info(f"Fetching Wikipedia article for topic: {WIKIPEDIA_TOPIC}")

try:
    if not WIKIPEDIA_TOPIC or not isinstance(WIKIPEDIA_TOPIC, str):
        raise ValueError("WIKIPEDIA_TOPIC must be a non-empty string")

    wiki_text = fetch_wikipedia_article(WIKIPEDIA_TOPIC)

    if not wiki_text or not wiki_text.strip():
        raise RuntimeError(f"No content returned for topic: {WIKIPEDIA_TOPIC}")

    logger.info(
        f"Fetched Wikipedia article successfully | "
        f"Topic='{WIKIPEDIA_TOPIC}' | "
        f"Characters={len(wiki_text)}"
    )

    # Optional: preview first few characters (safe for logs)
    logger.debug(
        f"Wikipedia preview:\n{wiki_text[:MAX_LOG_CHARS]}..."
    )

except Exception as e:
    logger.error(f"Failed to fetch Wikipedia article for '{WIKIPEDIA_TOPIC}': {e}")
    raise



In [ ]:
# ---------------- Config ----------------
MIN_SCRIPT_CHARS = 600     # quality guardrail
RETRY_DELAY_SEC = 2
MAX_RETRIES = 2
# ---------------------------------------

logger.info("Starting script prompt generation")

try:
    if not wiki_text or not wiki_text.strip():
        raise ValueError("wiki_text is empty or invalid")

    # Generate prompt
    HINGLISH_PROMPT = generate_script_prompt(wiki_text)
    logger.info("Script prompt generated successfully")

    if not HINGLISH_PROMPT or not HINGLISH_PROMPT.strip():
        raise RuntimeError("Generated prompt is empty")

except Exception as e:
    logger.error(f"Failed during prompt generation: {e}")
    raise

# ---------------- Script Generation ----------------
script = None
for attempt in range(1, MAX_RETRIES + 1):
    try:
        logger.info(f"Generating script (attempt {attempt}/{MAX_RETRIES})")

        script = generate_script(HINGLISH_PROMPT, gemini_model)

        if not script or not script.strip():
            raise RuntimeError("Generated script is empty")

        if len(script) < MIN_SCRIPT_CHARS:
            raise RuntimeError(
                f"Generated script too short ({len(script)} chars)"
            )

        logger.info(
            f"Script generated successfully | Characters={len(script)}"
        )
        break

    except Exception as e:
        logger.warning(f"Script generation failed: {e}")


In [ ]:
import logging

# ---------------- Config ----------------
MIN_SEGMENTS = 1
# ---------------------------------------

logger.info("Starting audio segment generation")

try:
    if not script or not isinstance(script, str) or not script.strip():
        raise ValueError("Script is empty or invalid")

    # ✅ Correct call (single argument)
    audio_segments = generate_audio_segments(script)

    if not audio_segments or not isinstance(audio_segments, list):
        raise RuntimeError("generate_audio_segments returned no audio segments")

    if len(audio_segments) < MIN_SEGMENTS:
        raise RuntimeError(
            f"Insufficient audio segments generated ({len(audio_segments)})"
        )

    logger.info(
        f"Audio segments generated successfully | Segments={len(audio_segments)}"
    )

except Exception as e:
    logger.error(f"Audio generation failed: {e}")
    raise


In [ ]:
import os
import logging
from google.colab import files

# ---------------- Config ----------------
OUTPUT_FILE = CONFIG.get("OUTPUT_FILENAME")
# ---------------------------------------

logger.info("Starting audio combination and export")

try:
    if not audio_segments or not isinstance(audio_segments, list):
        raise ValueError("audio_segments is empty or invalid")

    if not OUTPUT_FILE or not isinstance(OUTPUT_FILE, str):
        raise ValueError("CONFIG['OUTPUT_FILENAME'] is missing or invalid")

    # ✅ Correct function call (no extra kwargs)
    combine_and_export_audio(
        audio_segments,
        OUTPUT_FILE
    )

    # Verify file creation
    if not os.path.exists(OUTPUT_FILE):
        raise RuntimeError(f"Output file was not created: {OUTPUT_FILE}")

    file_size_mb = os.path.getsize(OUTPUT_FILE) / (1024 * 1024)

    logger.info(
        f"Audio file generated successfully | "
        f"File='{OUTPUT_FILE}' | "
        f"Size={file_size_mb:.2f} MB"
    )

    # 📥 Colab-only download (kept outside core logic)
    files.download(OUTPUT_FILE)

except Exception as e:
    logger.error(f"Audio export failed: {e}")
    raise